# United States Extreme Climate Events Analysis
## - Feature Engineering

Feature Engineering

Feature engineering (engenharia de recursos) é o processo de transformar dados brutos em recursos (features) (variáveis) que melhor representam o problema para algoritmos de machine learning, aumentando a precisão e eficiência do modelo.

- Create derived variables:
    - Annual frequency by state 
    - Moving average of events
    - Seasonality
    - Severity index (based on damages/deaths)
- Tools: Python (Pandas, NumPy)

Structure for your new notebook:

1. IMPORTS
import pandas as pd
import numpy as np

2. LOAD DATA
Load the cleaned data from step 01
df = pd.read_csv('path_to_your_cleaned_data.csv')

3. FEATURE 1: Annual frequency by state
Your code here...

4. FEATURE 2: Moving average
Your code here...
5. FEATURE 3: Seasonality
Your code here...

6. FEATURE 4: Severity index
Your code here...

7. SAVE ENRICHED DATA
df.to_csv('data_with_features.csv')

## Initial Setup and Data Loading

In [29]:
import pandas as pd 
import numpy as np

#Load the dataset

df = pd.read_csv('../data/processed/final_dataset.csv')

C:\Users\lucas\AppData\Local\Temp\ipykernel_40264\3368521713.py:6: DtypeWarning: Columns (0: FLOOD_CAUSE, 1: TOR_OTHER_WFO, 2: TOR_OTHER_CZ_STATE, 3: TOR_OTHER_CZ_NAME) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/final_dataset.csv')


In [31]:
import pandas as pd
import numpy as np

def convert_damage_value(value):
    # 1. Handle actual NaN (null) values
    if pd.isna(value):
        return 0.0
    
    # 2. Handle string values
    if isinstance(value, str):
        value = value.upper().replace('$', '').strip()
        
        # Check if the string is empty after stripping spaces
        if value == '':
            return 0.0
            
        if 'K' in value:
            try:
                return float(value.replace('K', '')) * 1000
            except: return 0.0
        if 'M' in value:
            try:
                return float(value.replace('M', '')) * 1000000
            except: return 0.0
            
    # 3. Final attempt to convert to float
    try:
        return float(value)
    except (ValueError, TypeError):
        return 0.0

# Apply the updated function
df['DAMAGE_PROPERTY'] = df['DAMAGE_PROPERTY'].apply(convert_damage_value)

## Annual Frequency by State

In [ ]:
# 1. Create a proper Date column from BEGIN_YEARMONTH and BEGIN_DAY
# BEGIN_YEARMONTH is likely YYYYMM (e.g., 202305)
df['YEAR'] = df['BEGIN_YEARMONTH'].astype(str).str[:4].astype(int)
df['MONTH'] = df['BEGIN_YEARMONTH'].astype(str).str[4:].astype(int)

# Create a datetime column
df['BEGIN_DATE'] = pd.to_datetime(dict(year=df['YEAR'], 
                                       month=df['MONTH'], 
                                       day=df['BEGIN_DAY']))

# 2. Calculate Annual Frequency by State
# This counts how many events happened in that State during that specific Year
df['annual_freq_state'] = df.groupby(['STATE', 'YEAR'])['STATE'].transform('count')

## Moving Average

In [33]:
# Sorting is mandatory for moving averages
df = df.sort_values('BEGIN_DATE')

# 7-event rolling average for property damage
df['moving_avg_damage'] = df['DAMAGE_PROPERTY'].rolling(window=7, min_periods=1).mean()

KeyError: 'BEGIN_DATE'